# SAGE — Multi-Crop Vision AI Inference API (Google Colab Deployment)
This notebook deploys the SAGE FastAPI backend on Google Colab with GPU acceleration,
using **Qwen/Qwen2.5-VL-3B-Instruct** base model and fine-tuned LoRA adapters from **IDK-Chaitanya/SAGE5Class-cp**.

### Step 1: Install Requirements & Tunneling Client

In [ ]:
# Clone repository if not already present
import os
if not os.path.exists("Sage-backend"):
    !git clone https://github.com/YOUR_USERNAME/Sage-backend.git
    %cd Sage-backend

!pip install -q -r requirements.txt
!curl -s -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

### Step 2: Hugging Face Authentication via Colab Secrets

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("✅ HF_TOKEN loaded from Google Colab userdata secrets!")
except Exception as e:
    print("Colab secrets not set. Set HF_TOKEN in environment or prompt:")
    import getpass
    os.environ["HF_TOKEN"] = getpass.getpass("Enter HF_TOKEN: ")

### Step 3: Check GPU Status & Verify Checkpoints

In [ ]:
!nvidia-smi
!python scripts/verify_checkpoints.py

### Step 4: Pre-load Models & Run Local Smoke Test

In [ ]:
from app.models.model_manager import get_model_manager
from app.models.crop_registry import crop_registry
from app.inference.predictor import get_predictor
import io
from PIL import Image

manager = get_model_manager()
manager.load_base_model()

# Multimodal smoke test with sample image and custom prompt
img = Image.new("RGB", (300, 300), color=(50, 150, 50))
buf = io.BytesIO()
img.save(buf, format="JPEG")

predictor = get_predictor()
prompt = "Identify the disease affecting this corn plant."
res = predictor.predict(crop_id="corn", image_bytes=buf.getvalue(), prompt=prompt)
print("Multimodal smoke test result:", res.model_dump())


### Step 5: Start FastAPI Server & Expose via Cloudflare Tunnel

In [ ]:
import subprocess
import time
import re
import requests

# Launch FastAPI server in background
server_proc = subprocess.Popen(["python", "scripts/start_server.py"])
time.sleep(5)

# Launch Cloudflare Tunnel
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for _ in range(25):
    line = tunnel_proc.stdout.readline()
    if not line:
        time.sleep(1)
        continue
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

print("=" * 60)
print(f"🚀 PUBLIC API URL: {public_url}")
print(f"📖 SWAGGER DOCS:  {public_url}/docs")
print("=" * 60)